# La decisión más peligrosa
La selección / exclusión de datos (qué filas o casos se quitan del dataset, y por qué).
Por qué: borrar o filtrar ejemplos (especialmente de subgrupos minoritarios, casos “raros” o registros con valores faltantes) cambia la distribución del problema, puede ocultar fallos del modelo, introducir sesgos y producir conclusiones erradas que se ven “válidas” por las métricas. Es una decisión que suele hacerse sin registrarla y por eso es “invisible”.

In [3]:
# Preprocessing Decision Simulator (Colab / Jupyter)
# Pegar entero en Google Colab.
# Si falta ipywidgets: ejecutar en otra celda: !pip install ipywidgets

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from IPython.display import display, Markdown, clear_output

# ---- Generador de datos sintéticos ----
def generate_data(n=2000, minority_frac=0.2, seed=0, missing_frac=0.05, outlier_frac=0.02):
    rng = np.random.RandomState(seed)
    G = (rng.rand(n) < minority_frac).astype(int)   # grupo sensible: 0 mayoritario, 1 minoritario
    X1 = rng.normal(loc=0.0 + 1.0*G, scale=1.0, size=n)  # shift por grupo
    X2 = rng.normal(size=n)
    logits = -0.5 + 1.2 * X1 + 0.2 * G
    prob = 1 / (1 + np.exp(-logits))
    Y = (rng.rand(n) < prob).astype(int)
    df = pd.DataFrame({'X1': X1, 'X2': X2, 'G': G, 'Y': Y})
    # missing en X1
    miss_idx = rng.choice(n, size=int(missing_frac * n), replace=False)
    df.loc[miss_idx, 'X1'] = np.nan
    # outliers en X1
    out_idx = rng.choice(n, size=int(outlier_frac * n), replace=False)
    df.loc[out_idx, 'X1'] = df.loc[out_idx, 'X1'].fillna(0) + rng.normal(loc=10.0, scale=2.0, size=len(out_idx))
    return df

# ---- Preprocessing (y registro) ----
def apply_preprocessing(df_raw, exclude_minority=False, impute_missing=True, remove_outliers=False, add_leak=False):
    df = df_raw.copy()
    log = []
    initial_n = len(df)
    log.append({'step': 'initial_counts', 'details': df['G'].value_counts().sort_index().to_dict(), 'n_before': initial_n, 'n_after': initial_n})
    if exclude_minority:
        n_before = len(df)
        removed = df[df['G']==1].shape[0]
        df = df[df['G']!=1].reset_index(drop=True)
        log.append({'step': 'exclude_group', 'details': f'Removed G==1 count={removed}', 'n_before': n_before, 'n_after': len(df)})
    n_before = len(df)
    missing_before = df['X1'].isna().sum()
    if impute_missing:
        imp = SimpleImputer(strategy='median')
        df['X1'] = imp.fit_transform(df[['X1']])
        log.append({'step': 'impute_missing', 'details': f'Imputed median missing X1 (before_missing={missing_before})', 'n_before': n_before, 'n_after': len(df)})
    else:
        df = df.dropna(subset=['X1']).reset_index(drop=True)
        removed = n_before - len(df)
        log.append({'step': 'drop_missing', 'details': f'Dropped rows with missing X1 removed={removed}', 'n_before': n_before, 'n_after': len(df)})
    if remove_outliers:
        n_before = len(df)
        z = (df['X1'] - df['X1'].mean()) / (df['X1'].std() + 1e-9)
        mask = z.abs() <= 3.0
        removed = (~mask).sum()
        df = df[mask].reset_index(drop=True)
        log.append({'step': 'remove_outliers', 'details': f'Removed X1 outliers removed={int(removed)}', 'n_before': n_before, 'n_after': len(df)})
    if add_leak:
        df['leak'] = df['Y'] + np.random.normal(scale=0.1, size=len(df))
        log.append({'step': 'add_leak', 'details': 'Added leak feature correlated with Y (simulated)', 'n_before': len(df), 'n_after': len(df)})
    else:
        if 'leak' in df.columns:
            df = df.drop(columns=['leak'])
    log.append({'step': 'final_counts', 'details': df['G'].value_counts().sort_index().to_dict(), 'n_before': initial_n, 'n_after': len(df)})
    return df, log

# ---- Entrenamiento y métricas (global + por grupo) ----
def train_and_evaluate(df, features=None, test_size=0.25, seed=0):
    if features is None:
        features = ['X1','X2'] + (['leak'] if 'leak' in df.columns else [])
    X = df[features].values; y = df['Y'].values
    Xtr, Xte, ytr, yte, i_tr, i_te = train_test_split(X, y, np.arange(len(y)), test_size=test_size, random_state=seed, stratify=y)
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(Xtr); Xte = scaler.transform(Xte)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(Xtr, ytr)
    yprob = clf.predict_proba(Xte)[:,1]; ypred = clf.predict(Xte)
    auc = roc_auc_score(yte, yprob)
    acc = accuracy_score(yte, ypred)
    df_test = df.iloc[i_te].copy().reset_index(drop=True)
    df_test['y_prob'] = yprob; df_test['y_pred'] = ypred
    metrics = {'auc': auc, 'accuracy': acc, 'n_test': len(df_test)}
    group_metrics = {}
    for g in sorted(df['G'].unique()):
        sub = df_test[df_test['G']==g]
        if len(sub)==0:
            group_metrics[g] = {'n': 0, 'auc': None, 'accuracy': None, 'pos_rate_pred': None}
            continue
        gm_auc = roc_auc_score(sub['Y'], sub['y_prob']) if len(sub['Y'].unique())>1 else None
        gm_acc = accuracy_score(sub['Y'], sub['y_pred'])
        pos_rate = sub['y_pred'].mean()
        group_metrics[g] = {'n': len(sub), 'auc': gm_auc, 'accuracy': gm_acc, 'pos_rate_pred': pos_rate}
    pos_rates = [group_metrics[g]['pos_rate_pred'] if group_metrics[g]['pos_rate_pred'] is not None else 0 for g in group_metrics]
    fairness_gap = pos_rates[0] - pos_rates[1] if len(pos_rates)>=2 else None
    return metrics, group_metrics, fairness_gap, clf, scaler

# ---- Gráficos (un gráfico por figura) ----
def plot_feature_distribution(df_before, df_after, feature='X1'):
    fig1 = plt.figure(figsize=(6,3)); ax1 = fig1.add_subplot(1,1,1)
    ax1.hist(df_before[feature].dropna(), bins=30); ax1.set_title('Before: '+feature); ax1.set_xlabel(feature); ax1.set_ylabel('count')
    plt.show()
    fig2 = plt.figure(figsize=(6,3)); ax2 = fig2.add_subplot(1,1,1)
    ax2.hist(df_after[feature].dropna(), bins=30); ax2.set_title('After: '+feature); ax2.set_xlabel(feature); ax2.set_ylabel('count')
    plt.show()

def plot_metrics(metrics, group_metrics):
    fig = plt.figure(figsize=(6,3)); ax = fig.add_subplot(1,1,1)
    ax.bar(['auc','accuracy'], [metrics['auc'], metrics['accuracy']]); ax.set_ylim(0,1); ax.set_title('Overall metrics')
    plt.show()
    groups = sorted(group_metrics.keys()); pos_rates = [group_metrics[g]['pos_rate_pred'] if group_metrics[g]['pos_rate_pred'] is not None else 0 for g in groups]
    fig2 = plt.figure(figsize=(6,3)); ax2 = fig2.add_subplot(1,1,1)
    ax2.bar([str(g) for g in groups], pos_rates); ax2.set_ylim(0,1); ax2.set_title('Predicted positive rate by group (test)')
    plt.show()

# ---- Interactivo (ipywidgets) o demo estático si no está instalado ----
def interactive_simulator():
    try:
        import ipywidgets as widgets
    except Exception:
        widgets = None
    if widgets is None:
        clear_output(wait=True)
        display(Markdown("# Preprocessing Simulator — demo estático"))
        display(Markdown("ipywidgets no está disponible. Para activar interactividad en Colab: `!pip install ipywidgets` y re-ejecutar. Mostrando demo estático con valores: exclude_minority=True, impute_missing=False, remove_outliers=True."))
        df_raw = generate_data()
        df_proc, log = apply_preprocessing(df_raw, exclude_minority=True, impute_missing=False, remove_outliers=True, add_leak=False)
        metrics, group_metrics, gap, clf, scaler = train_and_evaluate(df_proc)
        display(Markdown("## Preprocessing log"))
        display(pd.DataFrame(log))
        display(Markdown("## Sample processed data")); display(df_proc.head(6))
        display(Markdown(f"## Metrics: AUC={metrics['auc']:.3f}, Accuracy={metrics['accuracy']:.3f}, Fairness gap={gap:.3f}"))
        display(Markdown("### Per-group metrics")); display(pd.DataFrame(group_metrics).T)
        plot_feature_distribution(df_raw, df_proc, 'X1'); plot_metrics(metrics, group_metrics)
        return
    # Build widgets
    clear_output(wait=True)
    display(Markdown("# Preprocessing Decision Simulator (interactive)"))
    display(Markdown("Toggles: Excluir grupo minoritario, Imputar missing (median), Remover outliers, Agregar leak. Luego presiona 'Aplicar'."))
    exclude = widgets.Checkbox(value=False, description='Excluir grupo minoritario (G==1)')
    impute = widgets.Checkbox(value=True, description='Imputar missing X1 (median)')
    outlier = widgets.Checkbox(value=False, description='Remover outliers |z|>3')
    leak = widgets.Checkbox(value=False, description='Agregar feature leak (correlada con Y)')
    seed = widgets.IntText(value=0, description='Seed')
    run = widgets.Button(description='Aplicar preproc y entrenar', button_style='primary')
    controls = widgets.HBox([exclude, impute, outlier, leak, seed, run])
    out = widgets.Output()
    display(controls, out)
    def on_run(b):
        with out:
            clear_output(wait=True)
            df_raw = generate_data(seed=seed.value)
            df_proc, log = apply_preprocessing(df_raw, exclude_minority=exclude.value, impute_missing=impute.value, remove_outliers=outlier.value, add_leak=leak.value)
            metrics, group_metrics, gap, clf, scaler = train_and_evaluate(df_proc)
            display(Markdown("## Preprocessing log")); display(pd.DataFrame(log))
            display(Markdown("## Sample processed data")); display(df_proc.head(6))
            display(Markdown(f"## Metrics: AUC={metrics['auc']:.3f}, Accuracy={metrics['accuracy']:.3f}, Fairness gap={gap if gap is not None else 'N/A'}"))
            display(Markdown("### Per-group metrics")); display(pd.DataFrame(group_metrics).T)
            plot_feature_distribution(df_raw, df_proc, 'X1'); plot_metrics(metrics, group_metrics)
            display(Markdown("## Cómo explicarlo sencillamente (puntos a presentar):"))
            display(Markdown("1) Mostrar el *Preprocessing log* — cuántas filas y qué tipo de filas se eliminaron.\n2) Mostrar las métricas por grupo — si la precisión o tasa positiva cambia mucho, explicar impacto.\n3) Mostrar distribución antes/después para que cualquiera vea qué datos se perdieron."))
    run.on_click(on_run)
    # run once with defaults
    on_run(None)

# Ejecutar simulador
interactive_simulator()


# Preprocessing Decision Simulator (interactive)

Toggles: Excluir grupo minoritario, Imputar missing (median), Remover outliers, Agregar leak. Luego presiona 'Aplicar'.

Output()